# sample-mlops-agent — CloudWatch GenAI Observability

Explores OTEL traces emitted by the deployed AgentCore Runtime into `aws/spans` and the CloudWatch GenAI Observability Dashboard.

## What's already configured (no setup needed)

| Setting | Value |
|---|---|
| `AGENT_OBSERVABILITY_ENABLED` | `true` |
| `OTEL_TRACES_EXPORTER` | `otlp` → X-Ray → `aws/spans` |
| `OTEL_LOGS_EXPORTER` | `otlp` → CloudWatch Logs (runtime log group) |
| `OTEL_TRACES_SAMPLER` | `always_on` |
| `OTEL_SEMCONV_STABILITY_OPT_IN` | `gen_ai_tool_definitions` |
| `OTEL_ATTRIBUTE_VALUE_LENGTH_LIMIT` | `4096` |
| `cloud.resource_id` | patched post-deploy via `PatchRuntimeOtel` Custom Resource |

## Span hierarchy

```
invoke_agent sample_mlops_agent  (root — one per invoke_agent_runtime call)
  ├── POST /invocations          (Claude API HTTP call — botocore auto-instrumented)
  ├── S3.ListObjectsV2           (S3 workspace sync — botocore auto-instrumented)
  ├── S3.GetObject
  └── S3.PutObject
```

The root span is created in `agent/main.py` with GenAI semantic convention attributes.
Child spans are auto-instrumented by the OTEL botocore instrumentor.

In [ ]:
# Cell 1 — Imports and AWS session
import json
import os
import time
import uuid
from datetime import datetime, timezone

import boto3
from botocore.config import Config
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(dotenv_path=".env", override=False)

_config = Config(retries={"max_attempts": 10, "mode": "standard"})
session = boto3.Session(profile_name=os.environ.get("AWS_PROFILE", "default"))
AWS_REGION = session.region_name or "us-east-1"
AGENTCORE_STACK = os.environ.get("AGENTCORE_STACK", "sample-mlops-agent-agentcore")

_identity = session.client("sts", region_name=AWS_REGION, config=_config).get_caller_identity()
print(f"✓ AWS session")
print(f"  Account : ...{_identity['Account'][-4:]}")
print(f"  Region  : {AWS_REGION}")
print(f"  Role    : {_identity['Arn'].split('/')[-1]}")

In [ ]:
# Cell 2 — Resolve Runtime ARN from CloudFormation outputs
_cfn = session.client("cloudformation", region_name=AWS_REGION, config=_config)
_pages = _cfn.get_paginator("describe_stacks").paginate()
_stacks = [s for page in _pages for s in page.get("Stacks", [])]

_target = next((s for s in _stacks if AGENTCORE_STACK in s["StackName"]), None)
if not _target:
    raise RuntimeError(
        f"Stack '{AGENTCORE_STACK}' not found. "
        "Deploy first: cd cdk && npx cdk deploy sample-mlops-agent-agentcore"
    )

_outputs = {o["OutputKey"]: o["OutputValue"] for o in _target.get("Outputs", [])}
runtime_arn: str = _outputs["AgentRuntimeArn"]
agent_id: str = runtime_arn.rsplit("/", 1)[-1]
log_group = f"/aws/bedrock-agentcore/runtimes/{agent_id.split('-')[0]}"

print(f"Stack       : {_target['StackName']} ({_target['StackStatus']})")
print(f"RuntimeArn  : {runtime_arn}")
print(f"agent_id    : {agent_id}")
print(f"log_group   : {log_group}")

## Invoke the agent

Sends two test requests and collects `session_id` values for trace queries.

In [ ]:
# Cell 3 — Invoke agent (IAM SigV4)
TEST_INPUTS = [
    {"label": "list-jobs",    "message": "List my recent SageMaker training jobs and their status."},
    {"label": "mlflow-query", "message": "Show me the MLflow experiments and their latest run metrics."},
]

_ac = session.client("bedrock-agentcore", region_name=AWS_REGION, config=_config)
session_ids: list[str] = []
invoke_start_ts = int(time.time())

for i, test in enumerate(TEST_INPUTS):
    sid = str(uuid.uuid4())
    session_ids.append(sid)
    payload = json.dumps({"session_id": sid, "messages": [{"role": "user", "content": test["message"]}]}).encode()
    t0 = time.time()
    resp = _ac.invoke_agent_runtime(
        agentRuntimeArn=runtime_arn, runtimeSessionId=sid,
        payload=payload, qualifier="DEFAULT",
    )
    raw = b""
    for chunk in resp.get("response", b""):
        raw += chunk.get("chunk", {}).get("bytes", b"") if isinstance(chunk, dict) else chunk
    # BedrockAgentCoreApp streams responses in SSE format: each line is "data: {json}\n\n"
    # Only parse lines that start with "data: " and strip the prefix before decoding JSON.
    events = [json.loads(l[6:]) for l in raw.splitlines() if l.startswith(b"data: ")]
    text = "".join(e.get("delta", "") for e in events if e.get("type") == "TEXT_MESSAGE_CHUNK")
    print(f"[{i}] {test['label']} — {time.time()-t0:.1f}s — {sid[:8]}...")
    print(f"     {text[:120]}..." if len(text) > 120 else f"     {text}")

print(f"\n✓ {len(session_ids)} invocation(s) — invoke_start_ts={invoke_start_ts}")

## Confirm spans indexed

OTEL traces flow: AgentCore container → X-Ray (`https://xray.REGION.amazonaws.com/v1/traces`) → `aws/spans` CWL log group.
Indexing typically completes within **60–90 seconds**. The cell below retries until spans are visible.

In [ ]:
# Cell 5 — Confirm root spans visible in aws/spans
_logs = session.client("logs", region_name=AWS_REGION, config=_config)
window_start = invoke_start_ts - 60
window_end   = int(time.time()) + 60

confirmed: dict[str, bool] = {}
for sid in session_ids:
    found = False
    for attempt in range(4):
        qr = _logs.start_query(
            logGroupName="aws/spans",
            startTime=window_start, endTime=window_end,
            queryString=(
                f"fields spanId, name, traceId "
                f"| filter attributes.session.id = '{sid}' "
                f"| filter attributes.gen_ai.operation.name = 'invoke_agent' "
                f"| limit 1"
            ),
        )
        for _ in range(20):
            time.sleep(2)
            r = _logs.get_query_results(queryId=qr["queryId"])
            if r["status"] == "Complete":
                if r.get("results"):
                    found = True
                break
        if found:
            break
        if attempt < 3:
            print(f"  [{sid[:8]}] not yet visible (attempt {attempt+1}/4), waiting 20s...")
            time.sleep(20)
    confirmed[sid] = found
    print(f"  {'ok' if found else '!!'} [{sid[:8]}...] {'indexed' if found else 'NOT FOUND'}")

print(f"\n{sum(confirmed.values())}/{len(session_ids)} sessions confirmed in aws/spans")

## Query spans by type

The OTEL instrumentation produces these span names for the sample-mlops-agent:

| Span name | When emitted | Key attributes |
|---|---|---|
| `invoke_agent sample_mlops_agent` | Once per `invoke_agent_runtime` call | `session.id`, `gen_ai.system`, `gen_ai.operation.name`, `gen_ai.request.model` |
| `POST /invocations` | Each Claude API HTTP call | (botocore auto-instrumented) |
| `S3.*` | S3 workspace sync calls | (botocore auto-instrumented) |

In [ ]:
# Cell 6 — Query all span types for the test sessions
sid_filter = " or ".join(f"attributes.session.id = '{s}'" for s in session_ids)

queries = {
    "All spans (by name)": (
        f"fields name, traceId, spanId, attributes.session.id "
        f"| filter ({sid_filter}) "
        f"| stats count(*) as cnt by name "
        f"| sort cnt desc"
    ),
    "Root spans (invoke_agent)": (
        f"fields name, traceId, "
        f"  attributes.gen_ai.system as system, "
        f"  attributes.gen_ai.request.model as model, "
        f"  attributes.session.id as session "
        f"| filter attributes.gen_ai.operation.name = 'invoke_agent' "
        f"  and ({sid_filter})"
    ),
    "Claude API calls": (
        f"fields name, traceId, parentSpanId "
        f"| filter name = 'POST /invocations' "
        f"  and ({sid_filter})"
    ),
}

for title, query_str in queries.items():
    print(f"\n{'='*60}")
    print(f"Query: {title}")
    print('='*60)
    qr = _logs.start_query(
        logGroupName="aws/spans",
        startTime=window_start, endTime=window_end,
        queryString=query_str,
    )
    for _ in range(30):
        time.sleep(2)
        r = _logs.get_query_results(queryId=qr["queryId"])
        if r["status"] == "Complete":
            break
    rows = r.get("results", [])
    if not rows:
        print("  (no results)")
    for row in rows:
        vals = {f["field"]: f["value"] for f in row}
        print(f"  {vals}")

## GenAI Observability Dashboard

The AWS Console GenAI Observability Dashboard aggregates spans from `aws/spans` into charts for latency, token usage, tool call rates, and errors.

In [ ]:
# Cell 7 — Print console deep-links
console_base = f"https://{AWS_REGION}.console.aws.amazon.com"

dashboard_url = (
    f"{console_base}/cloudwatch/home?region={AWS_REGION}"
    f"#genai-observability:agentId={agent_id}"
)
xray_url = (
    f"{console_base}/xray/home?region={AWS_REGION}#/traces"
    f"?filter=annotation.session_id%3D%22{session_ids[0]}%22"
)
cwl_url = (
    f"{console_base}/cloudwatch/home?region={AWS_REGION}"
    f"#logsV2:log-groups/log-group/$252Faws$252Fspans"
)
app_logs_url = (
    f"{console_base}/cloudwatch/home?region={AWS_REGION}"
    f"#logsV2:log-groups/log-group/{log_group.replace('/', '$252F')}"
)

display(Markdown(f"""
### Console Links

| Link | URL |
|---|---|
| GenAI Observability Dashboard | [{dashboard_url}]({dashboard_url}) |
| X-Ray Traces (first session) | [{xray_url}]({xray_url}) |
| aws/spans CWL log group | [{cwl_url}]({cwl_url}) |
| Application Logs | [{app_logs_url}]({app_logs_url}) |
"""))

In [ ]:
# Cell 8 — Additional diagnostic queries
diag_queries = {
    "Slowest spans (top 10)": (
        f"fields name, duration, traceId, attributes.session.id as session "
        f"| filter ({sid_filter}) "
        f"| sort duration desc "
        f"| limit 10"
    ),
    "Error spans (excluding expected S3 session-restore failures)": (
        f"fields name, attributes.exception.message as error, traceId "
        f"| filter status.code = 'ERROR' "
        f"  and name != 'S3.GetObject' "
        f"  and ({sid_filter})"
    ),
    "S3 session-restore (expected errors on first turn)": (
        f"fields name, attributes.exception.type as exc_type, traceId "
        f"| filter status.code = 'ERROR' "
        f"  and name = 'S3.GetObject' "
        f"  and ({sid_filter})"
    ),
    "Spans per session": (
        f"fields attributes.session.id as session "
        f"| filter ({sid_filter}) "
        f"| stats count(*) as total by session"
    ),
}

for title, query_str in diag_queries.items():
    print(f"\n{'='*60}")
    print(f"Query: {title}")
    print('='*60)
    qr = _logs.start_query(
        logGroupName="aws/spans",
        startTime=window_start, endTime=window_end,
        queryString=query_str,
    )
    for _ in range(30):
        time.sleep(2)
        r = _logs.get_query_results(queryId=qr["queryId"])
        if r["status"] == "Complete":
            break
    rows = r.get("results", [])
    if not rows:
        print("  (no results)")
    for row in rows:
        vals = {f["field"]: f["value"] for f in row}
        print(f"  {vals}")

## Key Takeaways

- **`aws/spans`** is the CWL log group where X-Ray indexes OTEL spans; use CWL Insights to query them.
- **Root span** is `invoke_agent sample_mlops_agent` (one per `invoke_agent_runtime` call); filter on `attributes.session.id` to isolate a session.
- **Child spans** are auto-instrumented by botocore: `POST /invocations` (Claude API calls), `S3.*` (workspace sync).
- **S3.GetObject error spans are expected** on a session's first turn — `restore_session()` in `agent/main.py` attempts to fetch `claude.json` and `sdk_session_id` from S3, which don't exist yet. The botocore instrumentor marks these as ERROR before the exception handler runs. These are filtered out of the "Error spans" diagnostic query.
- **`cloud.resource_id`** must be present in `OTEL_RESOURCE_ATTRIBUTES` for the GenAI Observability Dashboard to show this agent — patched automatically by the `PatchRuntimeOtel` Custom Resource on deploy.
- **`OTEL_ATTRIBUTE_VALUE_LENGTH_LIMIT: 4096`** prevents large payloads from silently exceeding X-Ray's 64 KB segment limit.
- **LLO events** (input/output messages) flow to the runtime log group via ADOT LLOHandler — used by the evaluate API (see notebook 2).

## Online Evaluation Metrics (Evals tab)

The **Evals tab** in the UI surfaces AgentCore online-evaluation scores. An
`OnlineEvaluationConfig` (see the `sample-mlops-agent-evals` CDK stack) scores
live sessions with builtin judges (`Builtin.GoalSuccessRate`, `Builtin.Helpfulness`,
`Builtin.Correctness`) plus the custom **confirmation-gate** evaluator, publishing to
the CloudWatch namespace `Bedrock-AgentCore/Evaluations`.

Key facts (learned the hard way):
- The metric **name is the evaluator name** (there is no single `EvaluationScore` metric).
- Metrics are dimensioned by `service.name`, whose real value is `<runtime>.DEFAULT`
  (e.g. `sample_mlops_agent.DEFAULT`) — NOT `bedrock-agentcore`.
- Requires CloudWatch **Transaction Search** enabled (the evals stack turns it on).


In [ ]:
from datetime import datetime, timedelta
# Read the same online-eval metrics the Evals tab shows.
import boto3
cw = session.client('cloudwatch', region_name=AWS_REGION, config=_config)
runtime_service = 'sample_mlops_agent.DEFAULT'
for ev in ['Builtin.GoalSuccessRate','Builtin.Helpfulness','Builtin.Correctness']:
    r = cw.get_metric_data(
        MetricDataQueries=[{
            'Id':'m','Expression':(
                f'SEARCH(\'{{Bedrock-AgentCore/Evaluations,"service.name"}} '
                f'MetricName="{ev}" "service.name"="{runtime_service}"\', \'Average\', 86400)'),
            'ReturnData':True}],
        StartTime=datetime.utcnow()-timedelta(days=7), EndTime=datetime.utcnow())
    vals=[v for res in r['MetricDataResults'] for v in res.get('Values',[])]
    print(ev, '7d avg:', round(sum(vals)/len(vals),3) if vals else 'no data yet')
